In [1]:
import os
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

load_dotenv(override=True)

True

In [2]:
doc_loader = TextLoader("../data/text_files/machine_learning-2.txt")

raw_docs = doc_loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

chunks = splitter.split_documents(raw_docs)

chunks

[Document(metadata={'source': '../data/text_files/machine_learning-2.txt'}, page_content='Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus'),
 Document(metadata={'source': '../data/text_files/machine_learning-2.txt'}, page_content='data, and thus perform tasks without being explicitly programmed.[1] Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous'),
 Document(metadata={'source': '../data/text_files/machine_learning-2.txt'}, page_content='many previous machine learning approaches in performance.'),
 Document(metadata={'source': '../data/text_files/machine_learning-2.txt'}, page_content='Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (

In [3]:
embedding_model = HuggingFaceEmbeddings(model_name=os.getenv("HUGGING_FACE_EMBEDDING_MODEL"))

vector_store = FAISS.from_documents(chunks, embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13266.18it/s]


In [4]:
mmr_retriever = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k": 3}
)

In [5]:
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}

Answer:
""")

llm = init_chat_model(os.getenv("OLLAMA_CHAT_MODEL"))

In [6]:
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')

In [7]:
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

rag_chain = create_retrieval_chain(retriever=mmr_retriever, combine_docs_chain=document_chain)

In [8]:
query = {"input": "How does Machine learning work?"}

response = rag_chain.invoke(query)

print("✅ Answer: \n", response['answer'])

✅ Answer: 
 Machine learning (ML) is a field of study in artificial intelligence that involves the development and study of statistical algorithms. These algorithms function by learning from data and generalizing to unseen data. Through this process, models are trained to correctly predict the preassigned labels of a set of examples.
